# Chapter 2: Supervised Learning Algorithms

## Overview
This notebook explores core supervised learning algorithms for both classification and regression:
1. **Generalization Concepts**: Overfitting, underfitting, and model complexity.
2. **Instance-Based Methods**: $k$-Nearest Neighbors ($k$-NN Regressor & Classifier).
3. **Linear Models**: Linear Regression, Ridge ($L_2$), Lasso ($L_1$), Logistic Regression, and Linear Support Vector Classifiers.
4. **Tree-Based & Ensemble Models**: Decision Trees, Random Forests, and Gradient Boosted Decision Trees (GBDT).
5. **Kernelized Support Vector Machines (SVMs)** & **Multi-Layer Perceptrons (MLPs)**.
6. **Uncertainty Estimation**: Analyzing `decision_function` and `predict_proba`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing, load_breast_cancer, make_blobs
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Matplotlib global parameters
plt.rc("font", size=10)
plt.rc("axes", labelsize=11, titlesize=12)

## 1. $k$-Nearest Neighbors Regression

While $k$-NN classification uses majority voting, **$k$-NN regression** computes the average target value of the $k$ nearest training instances:

$$\hat{y} = \frac{1}{k} \sum_{i \in \mathcal{N}_k(x)} y_i$$

In [ ]:
# Generate synthetic 1D continuous dataset
np.random.seed(42)
X_wave = np.linspace(-3, 3, 100).reshape(-1, 1)
y_wave = np.sin(X_wave).ravel() + np.random.normal(0, 0.1, size=100)

X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_wave, y_wave, test_size=0.25, random_state=42
)

# Evaluate performance across different neighbor values
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
line_grid = np.linspace(-3, 3, 1000).reshape(-1, 1)

for n_neighbors, ax in zip([1, 3, 9], axes):
    knn_reg = KNeighborsRegressor(n_neighbors=n_neighbors)
    knn_reg.fit(X_train_w, y_train_w)

    train_r2 = knn_reg.score(X_train_w, y_train_w)
    test_r2 = knn_reg.score(X_test_w, y_test_w)

    ax.plot(line_grid, knn_reg.predict(line_grid), label="Model Prediction", color="teal", lw=2)
    ax.plot(X_train_w, y_train_w, "o", color="navy", alpha=0.6, label="Train Data")
    ax.plot(X_test_w, y_test_w, "v", color="crimson", alpha=0.8, label="Test Data")
    ax.set_title(f"k={n_neighbors} | Train $R^2$={train_r2:.2f} | Test $R^2$={test_r2:.2f}")
    ax.set_xlabel("Feature")
    ax.set_ylabel("Target")
    ax.grid(True, linestyle="--", alpha=0.5)

axes[0].legend()
plt.tight_layout()
plt.show()

## 2. Linear Models for Regression & Regularization

Linear models compute predictions using a weighted linear combination of input features:

$$\hat{y} = w_1 x_1 + w_2 x_2 + \dots + w_p x_p + b$$

### Regularization Variants
- **Ordinary Least Squares (OLS)**: Minimizes Mean Squared Error (MSE) without regularization.
- **Ridge Regression**: Adds an $L_2$ penalty to constrain weights ($\alpha \sum w_i^2$). Prevents overfitting by shrinking weights toward zero.
- **Lasso Regression**: Adds an $L_1$ penalty ($\alpha \sum |w_i|$). Enforces sparsity, driving non-essential feature weights to zero.

In [ ]:
# Load California Housing Dataset
housing = fetch_california_housing(as_frame=True)
X_h, y_h = housing.data, housing.target

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_h, y_h, test_size=0.2, random_state=42
)

# 1. OLS Linear Regression
ols = LinearRegression().fit(X_train_h, y_train_h)
print(f"OLS Train R²: {ols.score(X_train_h, y_train_h):.4f} | Test R²: {ols.score(X_test_h, y_test_h):.4f}")

# 2. Ridge Regression (L2 penalty)
ridge_10 = Ridge(alpha=10.0).fit(X_train_h, y_train_h)
print(f"Ridge (alpha=10) Train R²: {ridge_10.score(X_train_h, y_train_h):.4f} | Test R²: {ridge_10.score(X_test_h, y_test_h):.4f}")

# 3. Lasso Regression (L1 penalty)
lasso_001 = Lasso(alpha=0.01).fit(X_train_h, y_train_h)
print(f"Lasso (alpha=0.01) Train R²: {lasso_001.score(X_train_h, y_train_h):.4f} | Test R²: {lasso_001.score(X_test_h, y_test_h):.4f}")
print(f"Lasso Features Retained: {np.sum(lasso_001.coef_ != 0)} / {X_h.shape[1]}")

## 3. Linear Classification Models & Decision Boundaries

For binary classification, linear models use a decision boundary defined by a hyper-plane:

$$\hat{y} = \text{sign}(w_1 x_1 + w_2 x_2 + \dots + w_p x_p + b)$$

The regularizer hyperparameter $C$ acts as the inverse of regularization strength ($\text{small } C \implies \text{stronger regularization}$).

In [ ]:
cancer = load_breast_cancer()
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    cancer.data, cancer.target, stratify=cancer.target, random_state=42
)

for c_val in [0.01, 1.0, 100.0]:
    logreg = LogisticRegression(C=c_val, max_iter=10000).fit(X_train_c, y_train_c)
    print(f"Logistic Regression (C={c_val:<5}): "
          f"Train Accuracy = {logreg.score(X_train_c, y_train_c)*100:.2f}% | "
          f"Test Accuracy = {logreg.score(X_test_c, y_test_c)*100:.2f}%")

## 4. Decision Trees & Ensembles

Decision trees build hierarchical axis-aligned partition boundaries.
- **Overfitting Risk**: Deep decision trees easily memorize training noise. Controlling `max_depth` or `max_leaf_nodes` is critical for pruning.
- **Random Forests**: Averaging an ensemble of randomized, decorrelated decision trees reduces prediction variance.
- **Gradient Boosted Decision Trees (GBDT)**: Sequentially trains shallow trees to correct residual errors made by preceding estimators.

In [ ]:
# 1. Single Pruned Decision Tree
dt = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train_c, y_train_c)
print(f"Decision Tree (max_depth=4) Test Accuracy: {dt.score(X_test_c, y_test_c)*100:.2f}%")

# 2. Random Forest Ensemble
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train_c, y_train_c)
print(f"Random Forest (100 trees)   Test Accuracy: {rf.score(X_test_c, y_test_c)*100:.2f}%")

# 3. Gradient Boosted Decision Trees
gbdt = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gbdt.fit(X_train_c, y_train_c)
print(f"Gradient Boosting (GBDT)    Test Accuracy: {gbdt.score(X_test_c, y_test_c)*100:.2f}%")

# Inspect Top 5 Feature Importances from Random Forest
plt.figure(figsize=(8, 4))
top_indices = np.argsort(rf.feature_importances_)[-10:]
plt.barh(range(10), rf.feature_importances_[top_indices], color="teal")
plt.yticks(range(10), cancer.feature_names[top_indices])
plt.xlabel("Feature Importance")
plt.title("Top 10 Feature Importances (Random Forest)")
plt.tight_layout()
plt.show()

## 5. Kernelized Support Vector Machines (SVMs) & Neural Networks (MLP)

- **Radial Basis Function (RBF) Kernel SVM**: Maps inputs into infinite-dimensional spaces using distance metrics. Parameters `gamma` (kernel width) and $C$ (regularization) control model capacity.
- **Multi-Layer Perceptron (MLP)**: Non-linear neural architecture applying non-linear activations (e.g., ReLU) to combined linear transformations. Sensitive to feature scaling.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_c)
X_test_scaled = scaler.transform(X_test_c)

# Kernelized Support Vector Machine
svm = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)
svm.fit(X_train_scaled, y_train_c)
print(f"RBF Kernel SVM Test Accuracy (Scaled): {svm.score(X_test_scaled, y_test_c)*100:.2f}%")

# Multi-Layer Perceptron Classifier
mlp = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42)
mlp.fit(X_train_scaled, y_train_c)
print(f"MLP Neural Network Test Accuracy:      {mlp.score(X_test_scaled, y_test_c)*100:.2f}%")

## 6. Uncertainty Estimates from Classifiers

Most classification algorithms offer methods to quantify decision confidence:
1. `decision_function`: Returns raw confidence distance to the separating hyper-plane.
2. `predict_proba`: Returns normalized class posterior probability distributions $\in [0, 1]$.

In [ ]:
# Synthetic 2D Toy Dataset
X_blobs, y_blobs = make_blobs(n_samples=20, centers=2, cluster_std=1.5, random_state=42)
X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(X_blobs, y_blobs, test_size=5, random_state=42)

logreg_blobs = LogisticRegression().fit(X_tr_b, y_tr_b)

df_uncertainty = pd.DataFrame({
    "Raw Decision Function": logreg_blobs.decision_function(X_te_b),
    "Probability Class 0": logreg_blobs.predict_proba(X_te_b)[:, 0],
    "Probability Class 1": logreg_blobs.predict_proba(X_te_b)[:, 1],
    "Predicted Class": logreg_blobs.predict(X_te_b),
    "Actual Class": y_te_b
})

print("Uncertainty Metrics Output on Held-out Samples:")
print(df_uncertainty.to_string())